# Mamba / State-Space Models

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

A refresher on structured state-space models (SSMs) and **Mamba** — the sequence architecture that
scales **linearly** in sequence length and is a serious challenger to the Transformer for long sequences.

## 1. What & Why

A **state-space model (SSM)** maps an input sequence $x_t$ to an output $y_t$ through a hidden state $h_t$ that
evolves by a linear recurrence:

$$h_t = \bar{A}\, h_{t-1} + \bar{B}\, x_t, \qquad y_t = C\, h_t$$

It's the classic Kalman-filter / RNN form, but with the matrices *structured* so the recurrence can be run
either as a **sequential scan** (O(L) time, O(1) memory per step — great for inference) **or** as a single
**convolution** over the whole sequence (parallel during training). That duality is the whole trick.

**The problem it solves.** Self-attention costs **O(L²)** in compute and memory — quadratic in sequence
length L. Doubling the context quadruples the cost. For very long sequences (genomics, audio, long documents,
high-resolution signals) that wall hits hard. SSMs are **O(L)** and carry a fixed-size state, so cost grows
linearly and inference needs no growing KV cache.

**The catch with plain SSMs.** A linear time-invariant (LTI) SSM uses the *same* $\bar{A}, \bar{B}, C$ at every
step, so it can't decide what to remember based on *content* — it cannot do the selective copying / induction
tasks Transformers ace. **Mamba** (Gu & Dao, 2023) fixes this by making $\bar{B}, C$ and the step size $\Delta$
**functions of the input** ("selective" SSM, or S6). That breaks the convolution view, so Mamba uses a
hardware-aware **parallel associative scan** instead, keeping linear scaling.

**Reach for it when:** sequences are long (≫ a few thousand tokens), you want linear-time / streaming
inference, or attention's KV-cache memory is the bottleneck. **Don't** reach for it for short sequences where
attention is already cheap and far more battle-tested, or when you need the rich, well-understood tooling and
pretrained ecosystem around Transformers.

## 2. Mental Model

**An SSM is an RNN that can shapeshift into a CNN.**

Picture a tank with a fixed-size **state** $h$. Each step, the old state leaks/rotates by $\bar{A}$ and a bit
of the new input is poured in via $\bar{B}$; you read off the level through $C$. Because the update is *linear*,
unrolling the recurrence gives a closed form — $y$ is just the input convolved with a fixed kernel
$\bar{K} = (C\bar{B},\, C\bar{A}\bar{B},\, C\bar{A}^2\bar{B}, \dots)$. So:

```
          recurrent view              convolutional view
  x ─▶ [h]──▶[h]──▶[h]──▶ y     x ─▶ ( ⊛  fixed kernel K ) ─▶ y
       step by step,  O(L)         whole sequence at once, parallel
       O(1) state at inference     used during training (FFT/long conv)
```

Same math, two execution modes — train as a conv, infer as a recurrence.

**Where Mamba differs:** the "leak rate" and "how much input to pour in" are no longer constants — they're
**dials the input itself turns** every step. A large $\Delta$ means "pay attention, write to memory"; a tiny
$\Delta$ means "ignore this token, let the state coast." That input-dependence is what lets Mamba selectively
remember relevant tokens and forget filler — but it makes the kernel change every step, so the static
convolution no longer applies and you must scan.

## 3. Key Concepts

- **State $h_t$** — a fixed-size vector summarizing the past. Its size N (the *state dimension*, e.g. 16) is a
  capacity knob, independent of sequence length.
- **Continuous params $(A, B)$ and discretization** — the model is defined in continuous time and *discretized*
  with a step size $\Delta$: typically $\bar{A} = \exp(\Delta A)$, $\bar{B} \approx \Delta B$ (zero-order hold).
  $\Delta$ controls the effective timescale / memory length.
- **LTI vs selective (S6)** — *Linear Time-Invariant*: params fixed across time → expressible as a convolution.
  *Selective* (Mamba): $\Delta, B, C$ depend on the input → time-varying → needs a scan.
- **HiPPO / structured A** — initializing $A$ with HiPPO theory (e.g. S4) lets a fixed-size state compress long
  histories well. Mamba uses a simpler real diagonal $A$ and leans on selectivity instead.
- **Associative / parallel scan** — a prefix-scan (à la cumulative sum) that computes the recurrence in
  O(log L) depth on parallel hardware, so even the time-varying case trains fast.
- **Selective copying / induction** — the toy tasks that separate the models: they require content-based
  routing, which LTI SSMs fail and selective SSMs (and attention) pass.
- **Mamba block** — in practice the SSM is wrapped with a gated MLP and a local causal **conv1d** (short-range
  mixing) — roughly an SSM playing the role attention plays inside a Transformer block.
- **Lineage** — S4 → S5 → H3 → **Mamba (S6)** → Mamba-2 (the SSM↔attention "duality" / SSD framework).

## 4. Setup

The worked examples below use only **NumPy** and **PyTorch (CPU is fine)** — we implement the SSM recurrence
from scratch so the mechanics are visible. No GPU, no downloads, no API keys.

The *real* fused kernels live in the [`mamba-ssm`](https://github.com/state-spaces/mamba) package, but it
requires a CUDA build and isn't needed to understand the model. The last cell shows how you'd use it and is
**gated** so the notebook still runs without it.

In [1]:
# %pip install numpy torch   # already present in this environment
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)
print("numpy", np.__version__, "| torch", torch.__version__)

numpy 2.4.6 | torch 2.12.1


## 5. Worked Examples

### Example 1 — One SSM, two execution modes (recurrence ≡ convolution)

We build a tiny **LTI** SSM with state size N, run it as a sequential recurrence, then build its convolution
kernel $\bar{K} = (C\bar{B}, C\bar{A}\bar{B}, C\bar{A}^2\bar{B}, \dots)$ and convolve. The outputs match —
that's the duality that lets you *train as a conv, infer as a recurrence*.

In [2]:
def discretize(A, B, delta):
    """Zero-order-hold discretization of a continuous (A, B) with step delta."""
    A_bar = np.exp(delta * A)            # diagonal A -> elementwise exp
    B_bar = delta * B                    # 1st-order approx of (A^-1)(exp(dA)-I)B
    return A_bar, B_bar


def ssm_recurrent(x, A_bar, B_bar, C):
    """Sequential scan: O(L) steps, O(N) state. x: (L,) scalar input sequence."""
    N = A_bar.shape[0]
    h = np.zeros(N)
    ys = []
    for xt in x:
        h = A_bar * h + B_bar * xt       # h_t = A_bar h_{t-1} + B_bar x_t
        ys.append(C @ h)                 # y_t = C h_t
    return np.array(ys)


def ssm_conv(x, A_bar, B_bar, C):
    """Same SSM as a causal convolution with kernel K_k = C A_bar^k B_bar."""
    L = len(x)
    K = np.array([C @ (A_bar ** k * B_bar) for k in range(L)])  # impulse response
    y = np.array([np.dot(K[: t + 1][::-1], x[: t + 1]) for t in range(L)])
    return y, K


# A tiny stable LTI SSM (diagonal, decaying state) ----------------------------
N = 4
A = -np.array([0.2, 0.5, 1.0, 2.0])      # negative real parts -> stable / decaying
B = np.array([1.0, 1.0, 1.0, 1.0])
C = np.array([0.6, -0.4, 0.3, 0.1])
delta = 0.5
A_bar, B_bar = discretize(A, B, delta)

x = np.array([1.0, 0.0, 0.0, 2.0, -1.0, 0.0, 0.5, 0.0])  # input sequence (L=8)
y_rec = ssm_recurrent(x, A_bar, B_bar, C)
y_cnv, K = ssm_conv(x, A_bar, B_bar, C)

print("recurrent output :", np.round(y_rec, 4))
print("conv output      :", np.round(y_cnv, 4))
print("max abs diff     :", np.max(np.abs(y_rec - y_cnv)))
print("conv kernel K    :", np.round(K, 4), "  <- decays, so it's a learned soft memory")

recurrent output : [0.3    0.2251 0.1863 0.7637 0.2989 0.2848 0.4188 0.3651]
conv output      : [0.3    0.2251 0.1863 0.7637 0.2989 0.2848 0.4188 0.3651]
max abs diff     : 1.1102230246251565e-16
conv kernel K    : [0.3    0.2251 0.1863 0.1637 0.1487 0.1373 0.1276 0.1188]   <- decays, so it's a learned soft memory


The two modes agree to numerical precision. Note the kernel `K` **decays** — a stable SSM is a soft
exponential memory whose timescale is set by `A` and `delta`.

### Example 2 — Selectivity: why Mamba beats a plain (LTI) SSM

Now make the model **input-dependent**. We give each step its own $\Delta_t$ produced from the input, so the
state can be *gated*: a large $\Delta_t$ writes the token into memory and resets decay, a near-zero $\Delta_t$
tells the state to ignore the token and coast. This is the core of Mamba's **selective scan (S6)** — and it's
exactly what a fixed LTI kernel cannot express.

In [3]:
def selective_ssm(x, gate, A, B, C, delta_base=1.0):
    """Mamba-style time-varying SSM. delta_t depends on the input via `gate`.

    gate ~ 1  -> large delta_t -> write this token strongly into state
    gate ~ 0  -> delta_t ~ 0   -> A_bar ~ 1, B_bar ~ 0 -> state coasts (ignore token)
    """
    N = A.shape[0]
    h = np.zeros(N)
    ys = []
    for xt, g in zip(x, gate):
        delta_t = delta_base * g                 # <-- input-dependent step size
        A_bar = np.exp(delta_t * A)
        B_bar = delta_t * B
        h = A_bar * h + B_bar * xt
        ys.append(C @ h)
    return np.array(ys)


# Task: a 'marker' token (gate=1) says "remember the next value"; filler tokens
# (gate=0) should be ignored. A selective SSM latches the marked value; an LTI
# SSM (constant gate) cannot tell signal from filler.
A = -np.array([0.05, 0.05, 0.05, 0.05])   # slow decay so memory persists
B = np.ones(4)
C = np.array([1.0, 0.0, 0.0, 0.0])

x    = np.array([0.0, 7.0, 0.3, 0.1, 0.2, 0.0, 0.4])   # the 7.0 is the value to keep
gate = np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0])   # only step 1 is 'important'

y_sel = selective_ssm(x, gate,         A, B, C)         # selective: gate from input
y_lti = selective_ssm(x, np.ones_like(gate) * 0.3, A, B, C)  # LTI: constant gate

print("selective output:", np.round(y_sel, 3))
print("LTI output      :", np.round(y_lti, 3))
print()
print(f"selective state after the marked token stays ~ {y_sel[-1]:.2f} (latched the 7.0)")
print(f"LTI smears every token together -> final {y_lti[-1]:.2f}, can't isolate the signal")

selective output: [0. 7. 7. 7. 7. 7. 7.]
LTI output      : [0.    2.1   2.159 2.157 2.184 2.152 2.24 ]

selective state after the marked token stays ~ 7.00 (latched the 7.0)
LTI smears every token together -> final 2.24, can't isolate the signal


The selective model **latches** the marked value and lets filler coast past, while the LTI model blends
everything indiscriminately. Content-based gating is the qualitative jump from S4-style SSMs to Mamba.

### Example 3 — Linear vs quadratic: cost scaling vs attention

The practical headline. Count the dominant operations: an SSM does O(L·N) work (one fixed-size state update
per token); self-attention forms an L×L score matrix, O(L²·d). Watch the curves diverge as L grows.

In [4]:
N = 16    # SSM state dim
d = 64    # model width
print(f"{'seq len L':>10} | {'SSM  O(L*N)':>14} | {'attention O(L^2*d)':>20} | {'attn / ssm':>10}")
print("-" * 64)
for L in [256, 1024, 4096, 16384, 65536]:
    ssm_ops = L * N
    attn_ops = L * L * d
    print(f"{L:>10} | {ssm_ops:>14,} | {attn_ops:>20,} | {attn_ops / ssm_ops:>9.0f}x")

print("\nDoubling L doubles SSM cost but quadruples attention cost -> the gap explodes.")
print("Inference memory: SSM keeps a fixed O(N) state; attention's KV cache grows O(L).")

 seq len L |    SSM  O(L*N) |   attention O(L^2*d) | attn / ssm
----------------------------------------------------------------
       256 |          4,096 |            4,194,304 |      1024x
      1024 |         16,384 |           67,108,864 |      4096x
      4096 |         65,536 |        1,073,741,824 |     16384x
     16384 |        262,144 |       17,179,869,184 |     65536x
     65536 |      1,048,576 |      274,877,906,944 |    262144x

Doubling L doubles SSM cost but quadruples attention cost -> the gap explodes.
Inference memory: SSM keeps a fixed O(N) state; attention's KV cache grows O(L).


In [5]:
# OPTIONAL: the real fused-kernel Mamba block (needs a CUDA build of `mamba-ssm`).
# Gated so this notebook runs anywhere; shows the actual call shape.
import importlib.util

if importlib.util.find_spec("mamba_ssm") is not None and torch.cuda.is_available():
    from mamba_ssm import Mamba

    batch, length, dim = 2, 64, 16
    block = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2).to("cuda")
    out = block(torch.randn(batch, length, dim, device="cuda"))
    print("mamba_ssm available -> output shape:", tuple(out.shape))
else:
    print("mamba_ssm / CUDA not available — skipping the fused-kernel demo.")
    print("Install on a CUDA box with:  pip install mamba-ssm causal-conv1d")
    print("Usage shape:  Mamba(d_model=D)(x)  with x of shape (batch, length, D) -> same shape out.")

mamba_ssm / CUDA not available — skipping the fused-kernel demo.
Install on a CUDA box with:  pip install mamba-ssm causal-conv1d
Usage shape:  Mamba(d_model=D)(x)  with x of shape (batch, length, D) -> same shape out.


## 6. Gotchas & Pitfalls

- **Selectivity kills the convolution.** The moment $\Delta, B, C$ depend on the input, there is no single
  fixed kernel — you *must* use the associative scan. Don't expect to FFT-convolve a Mamba layer.
- **Stability lives in $A$.** Real parts of $A$ must be negative (so $|\bar{A}| < 1$) or the state explodes.
  Mamba parameterizes $A$ to stay negative and learns $\log\Delta$ for positivity — replicate that if you roll
  your own.
- **$\Delta$ is the timescale dial, and it's touchy.** Too small → the state barely updates (vanishing
  memory); too large → it forgets instantly. Initialization of $\Delta$ (the `dt_min`/`dt_max` range) matters a
  lot in practice.
- **The naive recurrence is slow on GPUs.** A Python/for-loop scan is sequential and memory-bound; the speed
  comes from the hardware-aware fused kernel that keeps the state in SRAM and never materializes it in HBM.
  Running the unfused reference at scale is what makes people wrongly conclude "Mamba is slow."
- **It's not pure attention-replacement parity.** On tasks needing precise long-range *retrieval* or
  in-context copying of arbitrary spans, pure SSMs can lag attention; many strong models are **hybrids**
  (mostly Mamba layers + a few attention layers), e.g. Jamba.
- **`mamba-ssm` needs CUDA + matching toolchain.** The pip package compiles custom kernels; CPU-only or
  mismatched CUDA/PyTorch versions are a common install headache. Prototype the math in NumPy (as above) first.
- **Bidirectional needs care.** SSMs are causal by construction; non-causal/vision use (e.g. Vision Mamba)
  runs scans in both directions and combines them — you can't just "unmask" like attention.

## 7. When to Use vs Alternatives

| Option | Train cost | Inference | Long-range | Best when |
|---|---|---|---|---|
| **Mamba / selective SSM** | O(L) scan | O(1) state per step, no KV cache | Strong, content-aware | Long sequences, streaming/low-latency inference, tight memory |
| **Transformer (attention)** | O(L²) | O(L) KV cache grows with context | Excellent, exact retrieval | Short–medium context, when ecosystem/pretrained weights matter |
| **Plain RNN / LSTM** | O(L), sequential | O(1) state | Weak (vanishing gradients) | Tiny models, legacy; mostly superseded |
| **S4 / LTI SSM** | O(L log L) conv | O(1) state | Strong on smooth signals | Continuous signals (audio, time series) without content routing |
| **Hybrid (Mamba + attention), e.g. Jamba** | mostly O(L) | mostly fixed state | Best of both | Long context **and** precise retrieval at production scale |

**Rules of thumb.** Short context (≤ a few k tokens)? Just use attention — it's cheap there, exact, and has the
richest tooling. Long context, audio, genomics, or latency-/memory-bound streaming? SSMs shine. Need both raw
long-context throughput *and* sharp in-context recall? Use a hybrid. Pure continuous signal with no need for
content-based gating? An LTI S4 may already be enough (and you keep the convolution speedup).

## 8. Resources

- **Mamba paper** — Gu & Dao, *Mamba: Linear-Time Sequence Modeling with Selective State Spaces* (2023): https://arxiv.org/abs/2312.00752
- **Reference implementation** — `state-spaces/mamba` on GitHub: https://github.com/state-spaces/mamba
- **The Annotated S4** — Rush & Karamcheti, builds a structured SSM from scratch (the best on-ramp to the math): https://srush.github.io/annotated-s4/
- **Mamba-2 / SSD** — Dao & Gu, *Transformers are SSMs: ... Structured State Space Duality* (2024): https://arxiv.org/abs/2405.21060
- **Jamba** — a production Mamba/attention hybrid (AI21): https://arxiv.org/abs/2403.19887
- **Visual walkthrough** — *A Visual Guide to Mamba and State Space Models* (Maarten Grootendorst): https://maartengrootendorst.com/blog/mamba/